In [ ]:
"""
Step 1.1 — Load Moon et al. PlanetScope LSP timing metrics for target
sites and years.

Loads the 7 raw timing metric GeoTIFFs per site per year into stacked
arrays with associated raster profiles. Metric layer paths and QA layer
paths follow the naming convention:

    {order}_{{year}}_{metric}.tif

where `order` is a leading integer, `{year}` is the 4-digit year, and
`metric` is the metric short name (e.g., OGI, 50PCGI, OGMx).

Fill value 32767 in metric layers is converted to NaN at load time.

QA layer loading is handled in Step 1.2.
"""

import json
import os
from pathlib import Path

import numpy as np
import rasterio
from rasterio.profiles import Profile

In [ ]:
# ---------------------------------------------------------------------------
# Config utilities
# ---------------------------------------------------------------------------

def build_config_path(config_filename, config_dir="configs"):
    """
    Build absolute path to a config file located under a config directory
    within the current working directory.

    Parameters
    ----------
    config_filename : str
        Filename of the config JSON
    config_dir : str, optional
        Subdirectory (relative to current working directory) where config
        files live. Defaults to "configs".

    Returns
    -------
    pathlib.Path
        Absolute path to the config file.
    """
    cwd = Path(os.getcwd())
    return cwd / config_dir / config_filename


def load_config(config_path):
    """
    Load a JSON config file from disk.

    Parameters
    ----------
    config_path : str or pathlib.Path
        Path to the JSON config file.

    Returns
    -------
    dict
        Parsed config as a Python dictionary.
    """
    with open(config_path, "r") as f:
        return json.load(f)


def compute_run_name(config):
    """
    Compute the run name as the concatenation of site_id, method, and
    run_number using underscores.

    Parameters
    ----------
    config : dict
        Config dict containing keys `site_id`, `method`, `run_number`.

    Returns
    -------
    str
        Run name of the form "{site_id}_{method}_{run_number}".
    """
    site_id = config["site_id"]
    method = config["method"]
    run_number = config["run_number"]
    return f"{site_id}_{method}_{run_number}"


def build_output_dir(config):
    """
    Create and return the output results directory for the current run.

    The directory is located under `{script_path}/results/{run_name}` where
    `run_name` is derived from site_id, method, and run_number.

    Parameters
    ----------
    config : dict
        Config dict containing keys `script_path`, `site_id`, `method`,
        `run_number`.

    Returns
    -------
    pathlib.Path
        Absolute path to the output results directory (created if
        missing).
    """
    output_path = Path(config["output_path"])
    run_name = compute_run_name(config)
    results_dir = output_path / run_name
    results_dir.mkdir(parents=True, exist_ok=True)
    return results_dir

In [ ]:
# ---------------------------------------------------------------------------
# Path parsing and resolution
# ---------------------------------------------------------------------------

def parse_layer_filename(filename):
    """
    Parse a Moon et al. LSP layer filename into (order, year, metric_name).

    The filename convention is `{order}_{{year}}_{metric}.tif`, parsed by
    splitting on underscores. The metric name may itself contain digits
    (e.g., "50PCGI").

    Parameters
    ----------
    filename : str
        Filename to parse. May be a full path or bare filename; only the
        basename is used.

    Returns
    -------
    tuple[int, str, str]
        Tuple of (order, year, metric_name) where:
        - order is the leading integer (int),
        - year is the 4-digit year (str),
        - metric_name is the metric short name with ".tif" stripped (str).

    Raises
    ------
    ValueError
        If the filename does not match the expected 3-part convention.
    """
    basename = Path(filename).name
    parts = basename.split("_")
    if len(parts) != 3:
        raise ValueError(
            f"Filename '{basename}' does not match expected format "
            f"'{{order}}_{{{year}}}_{{metric}}.tif' (got {len(parts)} parts)."
        )
    order_str, year_str, metric_with_ext = parts
    try:
        order = int(order_str)
    except ValueError:
        raise ValueError(
            f"Leading token '{order_str}' in '{basename}' is not an integer."
        )
    if not (year_str.isdigit() and len(year_str) == 4):
        raise ValueError(
            f"Year token '{year_str}' in '{basename}' is not a 4-digit year."
        )
    metric_name = metric_with_ext.replace(".tif", "")
    return order, year_str, metric_name


def resolve_layer_paths_for_year(layer_entries, data_path, year):
    """
    Expand {year} placeholders and prefix data root for a list of layer
    entries, for one specific year.

    Assumes data directory layout `{data_path}/{year}/{filename}`.

    Parameters
    ----------
    layer_entries : list[dict]
        List of layer entries from config, each having a "path" key with
        a filename that may contain the string "{year}" as year placeholder.
    data_path : str or pathlib.Path
        Root directory containing per-year subdirectories.
    year : str
        4-digit year string used to substitute "{year}" placeholders and to
        select the year subdirectory.

    Returns
    -------
    list[dict]
        New list of layer entries with "path" set to the absolute resolved
        path. Original entries are not modified.
    """
    root = Path(data_path) / str(year)
    resolved = []
    for entry in layer_entries:
        new_entry = dict(entry)
        raw_path = Path(new_entry["path"])
        if raw_path.is_absolute():
            resolved_path = str(raw_path).replace("{year}", str(year))
        else:
            resolved_path = str(root / raw_path).replace("{year}", str(year))
        new_entry["path"] = resolved_path
        resolved.append(new_entry)
    return resolved


In [ ]:
# ---------------------------------------------------------------------------
# Metric stack loading (single year)
# ---------------------------------------------------------------------------

def _assert_profiles_match(profile_a, profile_b, layer_a_path, layer_b_path):
    """
    Assert that two rasterio profiles have matching shape, CRS, and
    transform.

    Parameters
    ----------
    profile_a, profile_b : dict-like
        Rasterio profile dictionaries.
    layer_a_path, layer_b_path : str
        File paths corresponding to the two profiles, used in error
        messages.

    Raises
    ------
    ValueError
        If width, height, CRS, or transform differ between the profiles.
    """
    checks = [
        ("width", profile_a["width"], profile_b["width"]),
        ("height", profile_a["height"], profile_b["height"]),
        ("crs", profile_a["crs"], profile_b["crs"]),
        ("transform", profile_a["transform"], profile_b["transform"]),
    ]
    mismatches = [name for name, a, b in checks if a != b]
    if mismatches:
        raise ValueError(
            f"Profile mismatch between '{layer_a_path}' and '{layer_b_path}': "
            f"{mismatches} differ."
        )


def load_metric_stack(metric_layers, config):
    """
    Load a set of Moon et al. LSP timing-metric GeoTIFFs for one site and
    one year into a stacked float32 array.

    Layers are sorted by their `order` integer (parsed from filename)
    before stacking to guarantee a consistent feature-column order across
    calls. All layers must share the same width, height, CRS, and
    transform; a mismatch raises ValueError. Values equal to `fill_value`
    are converted to NaN.

    Parameters
    ----------
    metric_layers : list[dict]
        List of layer entries, each containing at minimum a resolved
        absolute "path" to a GeoTIFF.
    fill_value : int, optional
        Sentinel value used in the source rasters to mark missing
        pixels. Converted to NaN in the returned stack. Defaults to
        32767 (Moon et al. product fill value).

    Returns
    -------
    tuple[np.ndarray, list[str], rasterio.profiles.Profile]
        - stack : ndarray of shape (H, W, F), dtype float32, with
          `fill_value` replaced by NaN. F is the number of layers,
          ordered by the parsed `order` integer.
        - metric_names : list of str of length F, matching the stacking
          order; names are taken directly from parsed filenames.
        - profile : the rasterio Profile of the first layer (all layers
          share the same profile).

    Raises
    ------
    FileNotFoundError
        If any layer path does not exist on disk.
    ValueError
        If layers disagree on width, height, CRS, or transform, or if a
        filename does not match the expected convention.
    """
    # Parse and sort by order.
    parsed = []
    for entry in metric_layers:
        path = entry["path"]
        if not Path(path).exists():
            raise FileNotFoundError(f"Metric layer not found: {path}")
        order, year_str, metric_name = parse_layer_filename(path)
        parsed.append((order, year_str, metric_name, path))
    parsed.sort(key=lambda item: item[0])

    bands = []
    metric_names = []
    profile = None
    reference_path = None

    for order, year_str, metric_name, path in parsed:
        with rasterio.open(path) as src:
            band = src.read(1).astype(np.float32)
            layer_profile = src.profile

        if profile is None:
            profile = layer_profile
            reference_path = path
        else:
            _assert_profiles_match(profile, layer_profile, reference_path, path)

        band[band == config['qa']['fill_value']] = np.nan
        bands.append(band)
        metric_names.append(metric_name)

    stack = np.stack(bands, axis=-1)
    return stack, metric_names, profile

In [ ]:
# ---------------------------------------------------------------------------
# Metric stack loading (multi-year)
# ---------------------------------------------------------------------------

def _assert_cross_year_profiles_match(profiles_by_year):
    """
    Assert that raster profiles are consistent across years.

    All profiles must share the same width, height, CRS, and transform.

    Parameters
    ----------
    profiles_by_year : dict[str, rasterio.profiles.Profile]
        Mapping from year string to raster profile of that year's stack.

    Raises
    ------
    ValueError
        If any two years disagree on width, height, CRS, or transform.
    """
    years = list(profiles_by_year.keys())
    if len(years) < 2:
        return
    reference_year = years[0]
    reference_profile = profiles_by_year[reference_year]
    for year in years[1:]:
        _assert_profiles_match(
            reference_profile,
            profiles_by_year[year],
            layer_a_path=f"year={reference_year}",
            layer_b_path=f"year={year}",
        )


def load_metric_stack_multi(config):
    """
    Load Moon et al. LSP timing-metric stacks for one site across
    multiple years, keyed by year.

    Wraps `load_metric_stack` in a per-year loop, resolving layer paths
    for each year from the config's `data_path`, `years`, and
    `metric_layers` fields. Verifies that raster profiles are consistent
    across years and raises if not.

    Parameters
    ----------
    config : dict
        Config dict containing at minimum:
        - "data_path" : str, root data directory
        - "years"     : list[str], target years to load
        - "metric_layers" : list[dict], each with a "path" template that
          may contain "{year}" placeholder
    fill_value : int, optional
        Sentinel value to convert to NaN in the returned stacks.
        Defaults to 32767.

    Returns
    -------
    dict[str, tuple[np.ndarray, list[str], rasterio.profiles.Profile]]
        Mapping from year (str) to (stack, metric_names, profile) tuple
        as produced by `load_metric_stack`.

    Raises
    ------
    FileNotFoundError
        If any metric layer file is missing for any requested year.
    ValueError
        If layers within a year disagree on profile, if years disagree
        on profile, or if a filename does not match the expected
        convention.
    """
    years = [str(y) for y in config["years"]]
    data_path = config["data_path"]
    metric_layers_template = config["metric_layers"]

    stacks_by_year = {}
    profiles_by_year = {}
    for year in years:
        resolved_layers = resolve_layer_paths_for_year(
            metric_layers_template, data_path, year
        )
        stack, metric_names, profile = load_metric_stack(
            resolved_layers, config
        )
        stacks_by_year[year] = (stack, metric_names, profile)
        profiles_by_year[year] = profile

    _assert_cross_year_profiles_match(profiles_by_year)
    return stacks_by_year

In [19]:
"""
Step 1.2 — Build per-year QA masks for Moon et al. PlanetScope LSP
data.

QA masks are boolean arrays identifying pixels that are valid for
downstream clustering analysis. Each mask is constructed from one or
more QA layers, combined via the config's `qa_logic` field ("AND" or
"OR"). Pixels with fill value 32767 in any QA layer are treated as
invalid.

Retention statistics for each year are computed and added to the run
report JSON.
"""

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import rasterio

In [20]:
# ---------------------------------------------------------------------------
# Report file utilities
# ---------------------------------------------------------------------------

def initialize_report(config, output_dir):
    """
    Initialize (or reuse) the run report JSON file for this pipeline
    execution.

    If the report file already exists at the target location, it is
    loaded and returned unmodified so downstream steps can append. If
    it does not exist, a new report is created with run metadata (site
    name, method, run number, run name, timestamps) and no step
    entries.

    Parameters
    ----------
    config : dict
        Config dict containing at minimum "site_name", "method",
        "run_number", "report" (filename).
    output_dir : str or pathlib.Path
        Directory where the report file lives (typically the results
        directory returned by `build_output_dir`).

    Returns
    -------
    pathlib.Path
        Absolute path to the report file. The file is guaranteed to
        exist after this call.
    """
    report_path = Path(output_dir) / config["report"]
    if report_path.exists():
        return report_path

    now_iso = datetime.now(timezone.utc).isoformat()
    initial = {
        "run_metadata": {
            "site_name": config["site_name"],
            "method": config["method"],
            "run_number": config["run_number"],
            "run_name": (
                f"{config['site_name']}_{config['method']}_{config['run_number']}"
            ),
            "created_at": now_iso,
            "updated_at": now_iso,
        },
    }
    with open(report_path, "w") as f:
        json.dump(initial, f, indent=2)
    return report_path


def update_report(report_path, section_name, content):
    """
    Read-modify-write update to the run report JSON file.

    Adds or overwrites a top-level section keyed by `section_name` and
    updates the `updated_at` timestamp under `run_metadata`.

    Parameters
    ----------
    report_path : str or pathlib.Path
        Path to the report file. Must exist (call `initialize_report`
        first).
    section_name : str
        Top-level key under which `content` is stored. Overwrites any
        existing section with the same name.
    content : dict
        JSON-serializable content for this section.

    Returns
    -------
    dict
        The full report dict after the update, for optional inspection.

    Raises
    ------
    FileNotFoundError
        If `report_path` does not exist.
    """
    report_path = Path(report_path)
    if not report_path.exists():
        raise FileNotFoundError(
            f"Report file not found: {report_path}. "
            f"Call initialize_report first."
        )
    with open(report_path, "r") as f:
        report = json.load(f)
    report[section_name] = content
    report["run_metadata"]["updated_at"] = datetime.now(timezone.utc).isoformat()
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)
    return report

In [ ]:
# ---------------------------------------------------------------------------
# QA layer loading
# ---------------------------------------------------------------------------

def load_qa_layers(qa_layers, config):
    """
    Load QA GeoTIFF layers for one site and one year into a dict of
    float32 arrays keyed by QA metric name.

    Fill values (32767) are converted to NaN so downstream mask logic
    can treat missing data uniformly via NaN checks.

    Parameters
    ----------
    qa_layers : list[dict]
        List of QA layer entries, each with resolved "path" and
        "valid_range" (list of two numeric bounds).
    fill_value : int, optional
        Sentinel value converted to NaN. Defaults to 32767.

    Returns
    -------
    dict[str, np.ndarray]
        Mapping from QA metric name (parsed from filename) to the
        corresponding float32 array with fill values converted to NaN.

    Raises
    ------
    FileNotFoundError
        If any QA layer file is missing.
    ValueError
        If any filename does not match the expected convention.
    """
    qa_arrays = {}
    for entry in qa_layers:
        path = entry["path"]
        if not Path(path).exists():
            raise FileNotFoundError(f"QA layer not found: {path}")
        _, _, qa_name = parse_layer_filename(path)
        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)
        arr[arr == config['qa']['fill_value']] = np.nan
        qa_arrays[qa_name] = arr
    return qa_arrays

# ---------------------------------------------------------------------------
# QA mask construction
# ---------------------------------------------------------------------------

def build_qa_mask(qa_arrays, qa_layers, logic="AND"):
    """
    Construct a boolean QA mask from loaded QA arrays and their
    configured valid ranges.

    A pixel passes an individual QA layer if the value is non-NaN and
    within the layer's `valid_range` (inclusive on both bounds). Layer
    results are combined via `logic`:

    - "AND" : pixel must pass every layer to be valid
    - "OR"  : pixel need pass only one layer

    Parameters
    ----------
    qa_arrays : dict[str, np.ndarray]
        Mapping from QA metric name to float32 array (as produced by
        `load_qa_layers`).
    qa_layers : list[dict]
        List of QA layer entries with resolved "path" and
        "valid_range". Filenames are re-parsed to key into `qa_arrays`.
    logic : {"AND", "OR"}
        Combination logic. Defaults to "AND".

    Returns
    -------
    np.ndarray
        Boolean array of shape (H, W). True where the pixel passes the
        combined QA criteria.

    Raises
    ------
    ValueError
        If `logic` is not "AND" or "OR", or if a QA layer's parsed
        name is missing from `qa_arrays`.
    """
    per_layer_masks = []
    for entry in qa_layers:
        _, _, qa_name = parse_layer_filename(entry["path"])
        if qa_name not in qa_arrays:
            raise ValueError(
                f"QA layer '{qa_name}' not found in loaded qa_arrays. "
                f"Available: {list(qa_arrays.keys())}"
            )
        arr = qa_arrays[qa_name]
        low, high = entry["valid_range"]
        layer_mask = (~np.isnan(arr)) & (arr >= low) & (arr <= high)
        per_layer_masks.append(layer_mask)

    if logic == "AND":
        return np.logical_and.reduce(per_layer_masks)
    if logic == "OR":
        return np.logical_or.reduce(per_layer_masks)
    raise ValueError(f"Unsupported qa_logic: '{logic}'. Expected 'AND' or 'OR'.")

In [22]:
# ---------------------------------------------------------------------------
# Retention reporting
# ---------------------------------------------------------------------------

def report_qa_retention(mask, year):
    """
    Compute QA retention statistics for a single-year mask.

    Parameters
    ----------
    mask : np.ndarray
        Boolean QA mask of shape (H, W).
    year : str
        4-digit year string associated with the mask.

    Returns
    -------
    dict
        Retention statistics for this year:
        {
          "year": <str>,
          "n_total": <int>,
          "n_valid": <int>,
          "n_invalid": <int>,
          "valid_fraction": <float>,
          "invalid_fraction": <float>
        }
    """
    n_total = int(mask.size)
    n_valid = int(mask.sum())
    n_invalid = n_total - n_valid
    return {
        "year": str(year),
        "n_total": n_total,
        "n_valid": n_valid,
        "n_invalid": n_invalid,
        "valid_fraction": n_valid / n_total if n_total > 0 else 0.0,
        "invalid_fraction": n_invalid / n_total if n_total > 0 else 0.0,
    }


In [ ]:
# ---------------------------------------------------------------------------
# Single-year and multi-year QA mask wrappers
# ---------------------------------------------------------------------------

def build_qa_mask_for_year(config, year):
    """
    Build a QA mask for one site and one year.

    Resolves QA layer paths from config's `data_path`, `qa_layers`, and
    `qa_logic` fields, loads the layers, and constructs the boolean
    mask.

    Parameters
    ----------
    config : dict
        Config dict containing "data_path", "qa_layers", "qa_logic".
    year : str
        4-digit year string.
    fill_value : int, optional
        Sentinel converted to NaN during layer load. Defaults to 32767.

    Returns
    -------
    np.ndarray
        Boolean mask of shape (H, W). True where the pixel passes QA.

    Raises
    ------
    FileNotFoundError
        If any QA layer file is missing for this year.
    ValueError
        If QA logic is invalid or a QA layer name mismatches.
    """
    resolved_qa_layers = resolve_layer_paths_for_year(
        config['qa']['layers'], config["data_path"], year
    )
    qa_arrays = load_qa_layers(resolved_qa_layers, fill_value=config['qa']['fill_value'])
    mask = build_qa_mask(
        qa_arrays, resolved_qa_layers, logic=config['qa']['logic']
    )
    return mask


def build_qa_masks_multi(config):
    """
    Build per-year QA masks for one site across multiple years.

    Each year has its own QA layers and its own resulting mask; masks
    are not intersected or unioned across years. Returned as a dict
    keyed by year string.

    Parameters
    ----------
    config : dict
        Config dict containing "years", "data_path", "qa_layers",
        "qa_logic".
    fill_value : int, optional
        Sentinel converted to NaN during layer load. Defaults to 32767.

    Returns
    -------
    dict[str, np.ndarray]
        Mapping from year (str) to boolean mask of shape (H, W).

    Raises
    ------
    FileNotFoundError
        If any QA layer file is missing for any requested year.
    ValueError
        If QA logic is invalid or QA layer names mismatch within a
        year.
    """
    years = [str(y) for y in config["years"]]
    masks_by_year = {}
    for year in years:
        masks_by_year[year] = build_qa_mask_for_year(
            config, year, fill_value=config['qa_fill_value']
        )
    return masks_by_year

In [24]:
# ---------------------------------------------------------------------------
# Top-level orchestration for Step 1.2
# ---------------------------------------------------------------------------

def run_step_1_2(config, output_dir):
    """
    Execute Step 1.2: build per-year QA masks and update the run
    report with retention statistics.

    Initializes the report file if missing, computes per-year masks
    and retention stats, and writes the aggregated stats under the
    "step_1_2_qa_retention" section of the report.

    Parameters
    ----------
    config : dict
        Fully-loaded config dict.
    output_dir : str or pathlib.Path
        Results directory (as returned by `build_output_dir`).

    Returns
    -------
    dict[str, np.ndarray]
        Per-year QA masks keyed by year string, as produced by
        `build_qa_masks_multi`. Also updates the report on disk.
    """
    report_path = initialize_report(config, output_dir)
    masks_by_year = build_qa_masks_multi(config)

    retention_by_year = {
        year: report_qa_retention(mask, year)
        for year, mask in masks_by_year.items()
    }
    update_report(report_path, "step_1_2_qa_retention", retention_by_year)
    return masks_by_year

In [ ]:
"""
Step 1.3 — Stratified grid sampling of QA-passing pixels.

For each year, divides the raster into square cells of
`cell_size_px × cell_size_px` and draws `samples_per_cell` random
QA-passing pixels from each cell. Cells with zero QA-passing pixels
are skipped.

Random seed is derived per year as `random_seed_base + int(year)` to
guarantee independent (but reproducible) samples across years.

Outputs one CSV per year with sample metadata and metric values.
"""

from pathlib import Path

import numpy as np
import pandas as pd
import rasterio

# Assumes availability from Step 1.1 / 1.2 modules:
#   load_metric_stack, load_metric_stack_multi
#   load_qa_layers, build_qa_masks_multi, build_qa_mask_for_year
#   resolve_layer_paths_for_year
#   parse_layer_filename
#   initialize_report, update_report, build_output_dir


# ---------------------------------------------------------------------------
# Sampling core
# ---------------------------------------------------------------------------

def _cell_bounds(height, width, cell_size):
    """
    Yield (row_start, row_end, col_start, col_end) bounds for each
    grid cell tiling a raster of the given height and width.

    Edge cells at the right and bottom of the raster may be smaller
    than `cell_size` if height or width is not an integer multiple of
    `cell_size`.

    Parameters
    ----------
    height : int
        Raster height in pixels.
    width : int
        Raster width in pixels.
    cell_size : int
        Cell size in pixels (square cells).

    Yields
    ------
    tuple[int, int, int, int]
        (row_start, row_end, col_start, col_end) with end indices
        exclusive (Python slice convention).
    """
    for row_start in range(0, height, cell_size):
        row_end = min(row_start + cell_size, height)
        for col_start in range(0, width, cell_size):
            col_end = min(col_start + cell_size, width)
            yield row_start, row_end, col_start, col_end


def stratified_grid_sample(mask, cell_size, samples_per_cell, seed):
    """
    Draw stratified grid samples of pixel indices from a boolean mask.

    The raster is tiled into square cells of `cell_size × cell_size`.
    Within each cell, up to `samples_per_cell` mask-True pixels are
    drawn uniformly at random without replacement. Cells with fewer
    than `samples_per_cell` True pixels contribute all available True
    pixels. Cells with zero True pixels contribute nothing.

    Parameters
    ----------
    mask : np.ndarray
        Boolean array of shape (H, W). True where sampling is
        permitted (i.e., QA-passing pixels).
    cell_size : int
        Cell side length in pixels.
    samples_per_cell : int
        Number of samples to draw per cell (capped by the number of
        True pixels in that cell).
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    tuple[np.ndarray, np.ndarray, dict]
        - rows : ndarray of shape (N,), int64, sampled row indices
        - cols : ndarray of shape (N,), int64, sampled col indices
        - stats : dict with keys:
            "n_cells_total"    : total number of cells in the grid
            "n_cells_sampled"  : number of cells contributing >=1 sample
            "n_cells_empty"    : number of cells with zero True pixels
            "n_samples"        : total number of samples drawn
    """
    rng = np.random.default_rng(seed)
    height, width = mask.shape

    rows_out = []
    cols_out = []
    n_cells_total = 0
    n_cells_sampled = 0
    n_cells_empty = 0

    for row_start, row_end, col_start, col_end in _cell_bounds(
        height, width, cell_size
    ):
        n_cells_total += 1
        cell = mask[row_start:row_end, col_start:col_end]
        true_local_idx = np.flatnonzero(cell)
        if true_local_idx.size == 0:
            n_cells_empty += 1
            continue

        n_draw = min(samples_per_cell, true_local_idx.size)
        chosen_local = rng.choice(true_local_idx, size=n_draw, replace=False)

        cell_width = col_end - col_start
        local_rows, local_cols = np.divmod(chosen_local, cell_width)
        rows_out.append(local_rows + row_start)
        cols_out.append(local_cols + col_start)
        n_cells_sampled += 1

    if rows_out:
        rows = np.concatenate(rows_out).astype(np.int64)
        cols = np.concatenate(cols_out).astype(np.int64)
    else:
        rows = np.array([], dtype=np.int64)
        cols = np.array([], dtype=np.int64)

    stats = {
        "n_cells_total": int(n_cells_total),
        "n_cells_sampled": int(n_cells_sampled),
        "n_cells_empty": int(n_cells_empty),
        "n_samples": int(rows.size),
    }
    return rows, cols, stats


# ---------------------------------------------------------------------------
# Pixel-to-coordinate conversion
# ---------------------------------------------------------------------------

def pixel_to_xy(rows, cols, transform):
    """
    Convert (row, col) pixel indices to projected (x, y) coordinates
    using a rasterio affine transform.

    Coordinates returned are for the pixel center following the
    rasterio convention `transform * (col + 0.5, row + 0.5)`.

    Parameters
    ----------
    rows : np.ndarray
        Row indices, shape (N,).
    cols : np.ndarray
        Column indices, shape (N,).
    transform : affine.Affine
        Rasterio affine transform.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        (x, y) coordinate arrays of shape (N,), dtype float64.
    """
    xs, ys = rasterio.transform.xy(
        transform, rows.tolist(), cols.tolist(), offset="center"
    )
    return np.asarray(xs, dtype=np.float64), np.asarray(ys, dtype=np.float64)


# ---------------------------------------------------------------------------
# Sample table assembly
# ---------------------------------------------------------------------------

def _read_qa_values_at_pixels(qa_layer_entries, rows, cols, fill_value):
    """
    Read QA layer values at a set of pixel indices, returning a dict
    keyed by QA metric name.

    Fill values are converted to NaN so the sample table records
    missingness consistently. Called after sampling to attach raw QA
    values to sampled pixels.

    Parameters
    ----------
    qa_layer_entries : list[dict]
        Resolved QA layer entries (each with absolute "path").
    rows, cols : np.ndarray
        Pixel indices at which to sample values, shape (N,).
    fill_value : int or float
        Sentinel converted to NaN.

    Returns
    -------
    dict[str, np.ndarray]
        Mapping from QA metric name (parsed from filename) to sampled
        values, shape (N,), dtype float32.
    """
    values_by_name = {}
    for entry in qa_layer_entries:
        _, _, qa_name = parse_layer_filename(entry["path"])
        with rasterio.open(entry["path"]) as src:
            arr = src.read(1).astype(np.float32)
        arr[arr == fill_value] = np.nan
        values_by_name[qa_name] = arr[rows, cols]
    return values_by_name


def build_sample_table(
    metric_stack, metric_names, rows, cols, transform, year, qa_values_by_name
):
    """
    Assemble a per-year sample table (pandas DataFrame) from sampled
    pixels.

    Columns:
        row, col       : int64 pixel indices
        x, y           : float64 projected coordinates (pixel center)
        year           : str
        <metric_names> : float32 metric values in the order provided
        <qa_names>     : float32 QA values in the order provided

    Parameters
    ----------
    metric_stack : np.ndarray
        Metric stack of shape (H, W, F), as produced by
        `load_metric_stack`. Fill values already converted to NaN.
    metric_names : list[str]
        Feature names in the same order as the last axis of
        `metric_stack`.
    rows, cols : np.ndarray
        Sampled pixel indices, shape (N,).
    transform : affine.Affine
        Rasterio affine transform of the raster grid.
    year : str
        4-digit year string, written to every row.
    qa_values_by_name : dict[str, np.ndarray]
        QA values at sampled pixels, keyed by QA name; each value has
        shape (N,).

    Returns
    -------
    pandas.DataFrame
        Sample table of shape (N, 5 + F + len(qa_values_by_name)).
    """
    x, y = pixel_to_xy(rows, cols, transform)
    metric_values = metric_stack[rows, cols, :]

    data = {
        "row": rows,
        "col": cols,
        "x": x,
        "y": y,
        "year": [str(year)] * rows.size,
    }
    for i, name in enumerate(metric_names):
        data[name] = metric_values[:, i]
    for qa_name, qa_vals in qa_values_by_name.items():
        data[qa_name] = qa_vals

    return pd.DataFrame(data)


# ---------------------------------------------------------------------------
# Per-year and multi-year orchestration
# ---------------------------------------------------------------------------

def sample_year(config, year, mask, metric_stack, metric_names, profile):
    """
    Draw stratified grid samples for one year and assemble the sample
    table.

    Parameters
    ----------
    config : dict
        Config dict containing "sampling" (with method, cell_size_px,
        samples_per_cell, random_seed_base), "qa_layers",
        "qa_fill_value", "data_path".
    year : str
        4-digit year string.
    mask : np.ndarray
        Boolean QA mask for this year, shape (H, W).
    metric_stack : np.ndarray
        Metric stack for this year, shape (H, W, F).
    metric_names : list[str]
        Feature names matching the last axis of `metric_stack`.
    profile : rasterio.profiles.Profile
        Raster profile for this year (used to extract transform).

    Returns
    -------
    tuple[pandas.DataFrame, dict]
        (sample_table, sampling_stats). Stats dict includes cell
        coverage counts and total samples drawn (see
        `stratified_grid_sample`).

    Raises
    ------
    ValueError
        If `sampling.method` is not "stratified_grid".
    """
    sampling_cfg = config["sampling"]
    method = sampling_cfg.get("method", "stratified_grid")
    if method != "stratified_grid":
        raise ValueError(
            f"Unsupported sampling method: '{method}'. Only "
            f"'stratified_grid' is currently implemented."
        )

    cell_size = int(sampling_cfg["cell_size_px"])
    samples_per_cell = int(sampling_cfg["samples_per_cell"])
    base_seed = int(sampling_cfg["random_seed_base"])
    year_seed = base_seed + int(year)

    rows, cols, stats = stratified_grid_sample(
        mask, cell_size=cell_size, samples_per_cell=samples_per_cell,
        seed=year_seed,
    )
    stats["seed"] = year_seed

    resolved_qa_layers = resolve_layer_paths_for_year(
        config["qa_layers"], config["data_path"], year
    )
    qa_values_by_name = _read_qa_values_at_pixels(
        resolved_qa_layers, rows, cols, fill_value=config["qa_fill_value"]
    )

    df = build_sample_table(
        metric_stack=metric_stack,
        metric_names=metric_names,
        rows=rows,
        cols=cols,
        transform=profile["transform"],
        year=year,
        qa_values_by_name=qa_values_by_name,
    )
    return df, stats


def write_sample_table(df, output_dir, config, year):
    """
    Write a per-year sample table to CSV using the results filename
    template.

    The template is retrieved from
    `config["results"]["step_1_3"]["sampled_pixels_template"]` and
    the `{year}` placeholder is substituted.

    Parameters
    ----------
    df : pandas.DataFrame
        Sample table to write.
    output_dir : str or pathlib.Path
        Results directory.
    config : dict
        Config dict.
    year : str
        Year string used to substitute `{year}` in the filename
        template.

    Returns
    -------
    pathlib.Path
        Absolute path to the written CSV.
    """
    template = config["results"]["step_1_3"]["sampled_pixels_template"]
    filename = template.replace("{year}", str(year))
    out_path = Path(output_dir) / filename
    df.to_csv(out_path, index=False)
    return out_path


def run_step_1_3(
    config, output_dir, masks_by_year, stacks_by_year
):
    """
    Execute Step 1.3: stratified grid sampling per year, sample table
    assembly and CSV output, and report update.

    Parameters
    ----------
    config : dict
        Fully-loaded config dict.
    output_dir : str or pathlib.Path
        Results directory.
    masks_by_year : dict[str, np.ndarray]
        Per-year QA masks (from `run_step_1_2` or
        `build_qa_masks_multi`).
    stacks_by_year : dict[str, tuple[np.ndarray, list[str], rasterio.profiles.Profile]]
        Per-year metric stacks (from `load_metric_stack_multi`).

    Returns
    -------
    dict[str, pandas.DataFrame]
        Mapping from year (str) to sample DataFrame. Also writes one
        CSV per year to `output_dir` and appends
        "step_1_3_sampling" to the run report.
    """
    report_path = Path(output_dir) / config["report"]

    sample_tables = {}
    sampling_report = {}

    for year in [str(y) for y in config["years"]]:
        mask = masks_by_year[year]
        stack, metric_names, profile = stacks_by_year[year]

        df, stats = sample_year(
            config=config,
            year=year,
            mask=mask,
            metric_stack=stack,
            metric_names=metric_names,
            profile=profile,
        )
        out_path = write_sample_table(df, output_dir, config, year)

        sample_tables[year] = df
        sampling_report[year] = {
            **stats,
            "output_csv": str(out_path),
            "cell_coverage_fraction": (
                stats["n_cells_sampled"] / stats["n_cells_total"]
                if stats["n_cells_total"] > 0 else 0.0
            ),
        }

    update_report(report_path, "step_1_3_sampling", sampling_report)
    return sample_tables

In [1]:
"""
Step 1.4 — Compute derived timing features from sampled metric values.

Derived features are defined in the config's `derived_features` block
as explicit operand pairs and an operation code, avoiding formula
string parsing. Each entry has:

    {"name": <output_col>,
     "formula": <display_only_string>,
     "metric1": <lhs_column_name>,
     "metric2": <rhs_column_name>,
     "operation": <op_code>}

Only operation "-" (subtraction) is currently supported.

Outputs one CSV per year containing raw metrics, QA values, spatial
metadata, and derived features. Basic per-feature statistics (min,
max, mean, sd, n_valid, n_nan) are recorded in the run report.
"""

from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
# ---------------------------------------------------------------------------
# Derived feature computation
# ---------------------------------------------------------------------------

# NOTE only "-" operation for now on known metric columns
def compute_derived_feature(df, metric1, metric2, operation, allowed_names):
    """
    Compute a single derived feature from two operand columns and an
    operation code.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing `metric1` and `metric2` as columns.
    metric1 : str
        Name of the left-hand-side operand column.
    metric2 : str
        Name of the right-hand-side operand column.
    operation : str
        Operation code. Only "-" is currently supported.
    allowed_names : set[str] or list[str]
        Set of column names permitted as operands. Both `metric1` and
        `metric2` must be members.

    Returns
    -------
    pandas.Series
        Result of applying `operation` to `df[metric1]` and
        `df[metric2]`.

    Raises
    ------
    ValueError
        If `operation` is not supported, or if `metric1` or `metric2`
        is not in `allowed_names`.
    KeyError
        If `metric1` or `metric2` is not a column of `df`.
    """
    allowed_names_set = set(allowed_names)
    for name in (metric1, metric2):
        if name not in allowed_names_set:
            raise ValueError(
                f"Operand '{name}' not in allowed operand set. "
                f"Allowed: {sorted(allowed_names_set)}."
            )

    if operation == "-":
        return df[metric1] - df[metric2]
    # Future operations (never reached given the check above)
    raise ValueError(f"Operation dispatch fell through for '{operation}'.")


def add_derived_features(df, derived_features_config, metric_names):
    """
    Add derived feature columns to a copy of the input DataFrame.

    Iterates over `derived_features_config` entries, computing each
    derived feature via `compute_derived_feature` and appending it as
    a new column with the entry's "name".

    Parameters
    ----------
    df : pandas.DataFrame
        Input sample table (from Step 1.3) containing raw metric
        columns matching `metric_names`.
    derived_features_config : list[dict]
        List of derived-feature specs. Each dict must contain:
        "name", "metric1", "metric2", "operation". The optional
        "formula" field is ignored (display only).
    metric_names : list[str]
        Names of raw metric columns available as operands.

    Returns
    -------
    tuple[pandas.DataFrame, list[str]]
        - out_df : DataFrame copy with derived columns appended.
        - derived_names : list of derived column names in config
          order.

    Raises
    ------
    ValueError
        If any spec references an unsupported operation or an unknown
        operand column.
    KeyError
        If a spec's operand column is not present in `df`.
    """
    out_df = df.copy()
    derived_names = []
    for entry in derived_features_config:
        name = entry["name"]
        metric1 = entry["metric1"]
        metric2 = entry["metric2"]
        operation = entry["operation"]
        out_df[name] = compute_derived_feature(out_df, metric1, metric2, operation, metric_names)
        derived_names.append(name)
    return out_df, derived_names

# ---------------------------------------------------------------------------
# Per-feature statistics
# ---------------------------------------------------------------------------

def compute_feature_stats(series):
    """
    Compute basic descriptive statistics for a numeric series,
    ignoring NaN values.

    Parameters
    ----------
    series : pandas.Series or np.ndarray
        Numeric values to summarize.

    Returns
    -------
    dict
        Dictionary with keys "min", "max", "mean", "sd", "n_valid",
        "n_nan". Numeric values are Python floats/ints (JSON-safe).
        If all values are NaN, min/max/mean/sd are None.
    """
    arr = np.asarray(series, dtype=np.float64)
    n_total = arr.size
    n_nan = int(np.isnan(arr).sum())
    n_valid = n_total - n_nan
    if n_valid == 0:
        return {
            "min": None, "max": None, "mean": None, "sd": None,
            "n_valid": 0, "n_nan": n_nan,
        }
    return {
        "min": float(np.nanmin(arr)),
        "max": float(np.nanmax(arr)),
        "mean": float(np.nanmean(arr)),
        "sd": float(np.nanstd(arr, ddof=1)) if n_valid > 1 else 0.0,
        "n_valid": int(n_valid),
        "n_nan": n_nan,
    }

In [30]:
# ---------------------------------------------------------------------------
# File I/O
# ---------------------------------------------------------------------------

def write_combined_features_table(df, output_dir, config, year):
    """
    Write the combined feature table (raw + derived) to CSV using the
    config's Step 1.4 filename template.

    Parameters
    ----------
    df : pandas.DataFrame
        Combined feature table to write.
    output_dir : str or pathlib.Path
        Results directory.
    config : dict
        Config dict; expected key
        `config["results"]["step_1_4"]["combined_features_template"]`
        with a "{year}" placeholder.
    year : str
        Year string used to substitute "{year}" in the filename
        template.

    Returns
    -------
    pathlib.Path
        Absolute path to the written CSV.
    """
    template = config["results"]["step_1_4"]["combined_features_template"]
    filename = template.replace("{year}", str(year))
    out_path = Path(output_dir) / filename
    df.to_csv(out_path, index=False)
    return out_path


# ---------------------------------------------------------------------------
# Orchestration
# ---------------------------------------------------------------------------

def run_step_1_4(config, output_dir, sample_tables_by_year, metric_names,
                 update_report_fn):
    """
    Execute Step 1.4: compute derived duration features per year and
    write combined feature CSVs.

    Appends "step_1_4_derived_features" to the run report with
    per-year basic statistics (min, max, mean, sd, n_valid, n_nan) for
    each derived feature.

    Parameters
    ----------
    config : dict
        Fully-loaded config dict. Expected keys:
        - `derived_features` : list of derived-feature specs
        - `results.step_1_4.combined_features_template` : filename
          template with "{year}" placeholder
        - `report` : report filename
    output_dir : str or pathlib.Path
        Results directory (containing the report JSON).
    sample_tables_by_year : dict[str, pandas.DataFrame]
        Per-year sample tables from Step 1.3.
    metric_names : list[str]
        Names of raw metric columns in the sample tables. Used as the
        allowed operand set.
    update_report_fn : callable
        Function with signature `update_report_fn(report_path,
        section_name, content)` used to append this step's report
        section. Passed in to avoid cross-module import.

    Returns
    -------
    dict[str, pandas.DataFrame]
        Per-year combined feature tables (raw + derived), keyed by
        year string. Also writes one CSV per year and updates the run
        report.

    Raises
    ------
    ValueError
        If any derived-feature spec has an unsupported operation or
        unknown operand column.
    """
    report_path = Path(output_dir) / config["report"]
    derived_features_config = config["derived_features"]

    combined_tables = {}
    step_report = {}

    for year, df in sample_tables_by_year.items():
        combined_df, derived_names = add_derived_features(
            df, derived_features_config, metric_names
        )
        out_path = write_combined_features_table(
            combined_df, output_dir, config, year
        )

        derived_stats = {
            name: compute_feature_stats(combined_df[name])
            for name in derived_names
        }
        step_report[year] = {
            "derived_feature_names": derived_names,
            "derived_feature_stats": derived_stats,
            "output_csv": str(out_path),
            "n_rows": int(len(combined_df)),
        }
        combined_tables[year] = combined_df

    update_report_fn(report_path, "step_1_4_derived_features", step_report)
    return combined_tables